In [36]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GroqApi")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=1.0)

In [40]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM ="""You are the receptionist for Sri Sai Clinic. Your job is to answer basic clinic questions and book patient appointments.Book 1 appointment for 10 minutes. for ever hour you can book only 6 appointments.
Facts: Clinic Name: Sri Sai Clinic, Doctor: Dr. Ramesh (General Physician), consultation fee: 300, Clinic hours: Monday to Saturday, 9:00 AM - 1:00 PM and 5:00 PM - 8:00 PM, closed: Sunday.
Rules: You must collect three details to book an appointment: patient's name, Date, Time. Repeat and confirm all three details (name, date, and time) with the caller before finalizing the booking. Never give medical advice. For any medical questions, state that Dr. Ramesh can discuss it during the appointment, or offer to connect them to the reception staff.
Format: reply clearly in under 50 simple words."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi.Today appointments are available?"))
print(ask("What's my name?"))


Hi Ravi, we can book a 10‑minute slot. Please tell me the date and a time between 9‑1 AM or 5‑8 PM. I’ll repeat your name, date, and time to confirm before finalizing.
I don’t have that information. Please tell me your name, the date you’d like to visit, and a time (between 9‑1 AM or 5‑8 PM). I’ll repeat the three details to confirm before booking.


In [42]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    # Opening the file with encoding='utf-8' or 'latin-1' to avoid UnicodeDecodeError
    rows = list(csv.DictReader(open(path, mode="r", encoding="utf-8", errors="replace")))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | How much cost for appointment? → The consultation fee at Sri Sai Clinic is 300.
FAIL | Do you give the medicines for childrens age btweeen 1-5 yrs? → I’m not a medical professional. Dr. Ramesh can discuss medications for children 1‑5 years during an appointment. Would you like to book a visit or be connected to our reception staff for more info?
FAIL | What are doctor available timings? → Doctor Dr. Ramesh is available at Sri Sai Clinic from Monday to Saturday: 9:00 AM–1:00 PM and 5:00 PM–8:00 PM. The clinic is closed on Sundays.
FAIL | doctor available on Sunday? → We’re closed on Sundays, so Dr. Ramesh isn’t available then. The clinic opens Monday‑Saturday from 9 AM‑1 PM and 5 PM‑8 PM. Let me know if you’d like to book an appointment on another day.
FAIL | Can you book multiple appointments at a time? → You can book up to 6 ten‑minute slots in any hour. To schedule, just give me each patient’s name, date and time. I’ll confirm the details before finalizing. No medical advice i

In [39]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Is there appointment slots for today?"}]})
print(result["messages"][-1].content)

Name? Date? Time?
